In [ ]:
# Cell 1: Import libraries and keyword groups

# json is a standard Python library used to read the keyword data file.
import json

# requests is an external library used to communicate with the JobTech API.
import requests


# Load the keyword categories from the JSON file.
# Keeping the keywords outside the Python code makes them easier to update.
with open("keywords.json", "r", encoding="utf-8") as file:
    keyword_data = json.load(file)

# Convert the JSON lists into sets.
# Sets make it clear that the program checks whether a word belongs to a category.
ARCHITECTURE_KEYWORDS = set(keyword_data["architecture"])
EXCLUSION_KEYWORDS = set(keyword_data["exclude"])
ROLE_KEYWORDS = set(keyword_data["roles"])

In [66]:
# Cell 2: Parent and child classes

# Job is the parent class for the possible search classifications.
# It stores the keywords and provides a default category for unknown input.
class Job:
    def __init__(self, keywords):
        self.keywords = keywords

    def get_category(self):
        return "okänd"

    # __str__ makes it possible to use the object as readable text.
    def __str__(self):
        return self.get_category()


# ArchitectureJob inherits the attributes and methods from Job.
# It overrides get_category() with the construction/architecture category.
class ArchitectureJob(Job):
    def get_category(self):
        return "bygg/arkitektur"

In [67]:
# Cell 3: Classify construction and architecture searches

# Architect terms are role words that are allowed as search input.
def is_architect_term(word):
    cleaned_word = word.strip().lower()
    return cleaned_word in ROLE_KEYWORDS


# Check that every word entered by the user appears in the advertisement text.
def ad_contains_all_words(ad_text, search_words):
    text = ad_text.lower()

    for word in search_words:
        if word.strip().lower() not in text:
            return False

    return True


# Decide whether a complete advertisement looks like a construction job.
def classify_job_text(text):
    # Replace hyphens so terms such as IT-arkitekt can be inspected as words.
    words = text.lower().replace("-", " ").split()

    # Exclusion keywords remove unrelated advertisements from the construction search.
    has_exclusion_keyword = any(
        word in EXCLUSION_KEYWORDS for word in words
    )

    # A job is construction-related if it contains an architecture keyword or role.
    has_architecture_keyword = any(
        word in ARCHITECTURE_KEYWORDS or word in ROLE_KEYWORDS
        for word in words
    )

    if has_architecture_keyword and not has_exclusion_keyword:
        return "bygg/arkitektur"

    # Mixed or unrelated advertisements are not accepted as construction jobs.
    return "okänd"


# Classify the user's search words before the API request is made.
def classify_search_words(words):
    cleaned_words = [word.strip().lower() for word in words]

    has_architecture_keyword = any(
        word in ARCHITECTURE_KEYWORDS or word in ROLE_KEYWORDS
        for word in cleaned_words
    )

    if has_architecture_keyword:
        return ArchitectureJob(cleaned_words)

    # This object keeps the input available for the validation message in main().
    return Job(cleaned_words)

In [68]:
# Cell 4: Fetch and clean API data

# Public endpoint used to retrieve current job advertisements.
API_URL = "https://jobsearch.api.jobtechdev.se/search"


def fetch_jobs(search_words, limit=10):
    # The API expects the search terms as one text string.
    search_text = " ".join(search_words)

    parameters = {
        "q": search_text,
        "limit": limit,
    }

    # The API request can fail because of network problems, HTTP errors,
    # invalid JSON, or a changed response structure.
    try:
        response = requests.get(API_URL, params=parameters, timeout=10)
        response.raise_for_status()
        result = response.json()
        return result["hits"]
    except requests.exceptions.RequestException:
        print("Kunde inte hämta jobb från API:et.")
        return []
    except (ValueError, KeyError):
        print("API:et returnerade ett oväntat svar.")
        return []


def clean_job(job):
    # The API response contains nested dictionaries.
    # These variables make the fields easier to read below.
    description = job.get("description", {})
    employer = job.get("employer", {})
    workplace = job.get("workplace_address", {})
    occupation = job.get("occupation", {})
    employment_type = job.get("employment_type", {})
    working_hours = job.get("working_hours_type", {})

    # Return only the fields used by the filtering and output steps.
    # Missing fields receive an empty string instead of stopping the program.
    return {
        "id": job.get("id", ""),
        "webpage_url": job.get("webpage_url", ""),
        "headline": job.get("headline", ""),
        "short_description": description.get("needs", ""),
        "description": description.get("text", ""),
        "employer": employer.get("name", ""),
        "city": workplace.get("city", ""),
        "municipality": workplace.get("municipality", ""),
        "occupation": occupation.get("label", ""),
        "employment_type": employment_type.get("label", ""),
        "working_hours": working_hours.get("label", ""),
        "application_deadline": job.get("application_deadline", ""),
        "number_of_vacancies": job.get("number_of_vacancies", ""),
    }

In [69]:
# Cell 5: Build and print the final output

def build_final_output(cleaned_jobs, answer_jobs, selected_field):
    # Create a smaller result structure for printing.
    # The full cleaned API records are not printed to the user.
    return {
        "search_words": answer_jobs,
        "field": str(selected_field),
        "jobs": [
            {
                "headline": job.get("headline", ""),
                "employer": job.get("employer", ""),
                "city": job.get("city", ""),
                "webpage_url": job.get("webpage_url", ""),
            }
            for job in cleaned_jobs
        ],
    }


def print_result(result):
    # Print the search information first so the user can understand the result.
    print("Search words:", result["search_words"])
    print("Selected field:", result["field"])

    # enumerate() adds a readable result number starting at one.
    for i, job in enumerate(result["jobs"], start=1):
        print(f"\n{i}. {job['headline']}")
        print(f"   Employer: {job['employer']}")
        print(f"   City: {job['city']}")
        print(f"   Link: {job['webpage_url']}")

In [70]:
# Cell 6: Main construction-job search workflow

def main():
    # Repeat the input stage until the user provides an accepted search.
    while True:
        # Ask the user for one or two words and validate the input length.
        try:
            answer_job = input(
                "Välj arkitektroll eller ett keyword: "
            )
            answer_jobs = answer_job.strip().lower().split()

            if len(answer_jobs) == 0:
                raise ValueError("Vänligen ange minst ett nyckelord.")

            if len(answer_jobs) > 2:
                raise ValueError("Vänligen ange endast ett eller två nyckelord.")

        # These errors can occur if the user cancels or the notebook input is closed.
        except (EOFError, KeyboardInterrupt):
            print("Inmatningen avbröts.")
            return
        except ValueError as error:
            print(error)
            continue

        # Only construction and architecture-related searches are supported.
        selected_field = classify_search_words(answer_jobs)

        if str(selected_field) != "bygg/arkitektur":
            print(
                "Ange arkitekt, architecture, arkitektur, architect "
                "eller ett bygg-/projekteringsnyckelord."
            )
            continue

        # Leave the input loop after an accepted search has been provided.
        break

    # Ask the API for a small number of jobs for this prototype.
    jobs = fetch_jobs(answer_jobs, limit=25)

    if not jobs:
        print("Inga jobb hittades.")
        return

    # Convert raw API records into the simpler dictionaries used by the filter.
    cleaned_jobs = [clean_job(job) for job in jobs]
    filtered_jobs = []

    # Keep a job only when it contains all search words and is construction-related.
    for job in cleaned_jobs:
        text = (
            job.get("headline", "") + " " +
            job.get("description", "") + " " +
            job.get("employer", "") + " " +
            job.get("occupation", "")
        ).lower()

        job_field = classify_job_text(text)

        if (
            ad_contains_all_words(text, answer_jobs)
            and job_field == str(selected_field)
        ):
            filtered_jobs.append(job)

    print("Jobs after keyword filtering:", len(filtered_jobs))

    # Prepare and print only the fields relevant to the user.
    result = build_final_output(
        filtered_jobs,
        answer_jobs,
        selected_field
    )

    print_result(result)

In [71]:
# Cell 7: Run the program

# Run this cell after the previous cells have been executed in order.
main()

Ange arkitekt, architecture, arkitektur, architect eller ett bygg-/projekteringsnyckelord.
Jobs after keyword filtering: 4
Search words: ['bim']
Selected field: bygg/arkitektur

1. CAD-ritare till Metos Stockholm
   Employer: Intensogruppen AB
   City: None
   Link: https://arbetsformedlingen.se/platsbanken/annonser/31422257

2. Vi söker en VS- kalkylator till Rörteknik i Farsta AB
   Employer: Sparc Group AB (publ)
   City: Bandhagen
   Link: https://arbetsformedlingen.se/platsbanken/annonser/31482169

3. Mechanical Plant Design Engineer
   Employer: Hitachi Energy Sweden AB
   City: None
   Link: https://arbetsformedlingen.se/platsbanken/annonser/31418731

4. Senior Mechanical Plant Design Engineer
   Employer: Hitachi Energy Sweden AB
   City: None
   Link: https://arbetsformedlingen.se/platsbanken/annonser/31515346
